#数据清洗

In [ ]:
#数据挖掘脚本，挖掘数据后需要进入数据清洗

!pip install vllm -q

import pandas as pd
import json
import re
import os
import torch
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

# ==========================================
# 1. 配置参数 (Configuration)
# ==========================================
INPUT_CSV = "/kaggle/input/akkadian2english-dataset/publications.csv"
RAW_OUTPUT_CSV = "mined_raw_h100.csv"      # 原始挖掘结果 (带脏数据)
FINAL_OUTPUT_CSV = "train_ready_mined.csv" # 最终清洗好的训练数据

# H100 80G 专属: Qwen 2.5 72B (Int4)
# 显存占用: ~42GB, 上下文空间: ~38GB
MODEL_PATH = "Qwen/Qwen2.5-72B-Instruct-GPTQ-Int4"

# 批处理大小：每跑完 50 页写一次盘，防止崩溃丢失数据
BATCH_SIZE = 50

# 粗筛限制：设为 None 处理所有页面，或者设为整数 (如 5000) 进行测试
MAX_PAGES_TO_PROCESS = None

# ==========================================
# 2. 粗筛函数 (Coarse Filtering)
# ==========================================
def coarse_filter(csv_path):
    print("🧹 [Step 1] Loading and filtering publications.csv...")
    df = pd.read_csv(csv_path)

    # 1. 基础筛选：has_akkadian
    if df['has_akkadian'].dtype == 'O':
        mask = df['has_akkadian'].astype(str).str.lower() == 'true'
    else:
        mask = df['has_akkadian'] == True

    candidates = df[mask].copy()
    print(f"   -> Pages marked with 'has_akkadian': {len(candidates)}")

    # 2. 语言特征筛选 (English + Akkadian)
    # 目的：过滤掉德语(Seite, Abb.), 法语, 或纯图片页
    english_stops = {'the', 'and', 'of', 'to', 'in', 'that', 'is', 'with', 'for'}

    def is_promising(text):
        text = str(text).lower()
        if len(text) < 100: return False

        # 必须包含英语常用词 (>= 5次)
        words = re.findall(r'\w+', text)
        eng_count = sum(1 for w in words if w in english_stops)

        # 必须包含阿卡德语特征 (连字符单词 >= 2次)
        hyphen_count = len(re.findall(r'[a-z]{2,}-[a-z]{2,}', text))

        return eng_count >= 5 and hyphen_count >= 2

    candidates['is_promising'] = candidates['page_text'].apply(is_promising)
    candidates = candidates[candidates['is_promising']]

    if MAX_PAGES_TO_PROCESS:
        candidates = candidates.head(MAX_PAGES_TO_PROCESS)

    print(f"   -> Final candidates after regex filtering: {len(candidates)}")
    return candidates

# ==========================================
# 3. LLM 初始化 (vLLM Setup)
# ==========================================
def init_llm():
    print(f"🚀 [Step 2] Initializing {MODEL_PATH} on H100...")
    llm = LLM(
        model=MODEL_PATH,
        quantization="gptq",         # 显式指定 GPTQ
        dtype="float16",             # 计算精度
        gpu_memory_utilization=0.90, # 吃满显存
        max_model_len=8192,          # 长上下文
        trust_remote_code=True,
        enforce_eager=True           # 避免某些 CUDA Graph 错误
    )
    # 贪婪采样：Temperature=0 保证输出最稳定、格式最规范
    sampling_params = SamplingParams(temperature=0.0, max_tokens=2048)
    return llm, sampling_params

# ==========================================
# 4. Prompt 构建
# ==========================================
def construct_prompts(df):
    system_prompt = """You are an expert Assyriologist. Extract aligned (Akkadian Transliteration, English Translation) pairs from the OCR text.

STRICT RULES:
1. EXTRACT: Only Old Assyrian lines (e.g., 'a-wi-lim', 'i-na') and their English translations.
2. IGNORE: German text ('Seite', 'Band'), French, footnotes, headers, bibliographies.
3. REPAIR: Convert 'x x' or '...' to '<gap>'. Join broken hyphens.
4. FORMAT: Return a JSON List of objects: [{"akk": "...", "en": "..."}, ...]
5. If no valid pairs found, return [].
"""
    prompts = []
    for text in df['page_text']:
        # 截断过长文本，保留前 5000 字符 (通常包含主要内容)
        content = str(text)[:5000]

        # Qwen ChatML 格式
        prompt = f"""<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
Extract from this text:
{content}<|im_end|>
<|im_start|>assistant
"""
        prompts.append(prompt)
    return prompts

# ==========================================
# 5. 主挖掘循环 (Batch Processing)
# ==========================================
def run_mining(llm, sampling_params, prompts):
    print(f"⛏️ [Step 3] Starting Mining Loop (Total Pages: {len(prompts)})...")

    # 初始化 CSV 文件头
    if not os.path.exists(RAW_OUTPUT_CSV):
        pd.DataFrame(columns=['transliteration', 'translation']).to_csv(RAW_OUTPUT_CSV, index=False)

    total_pairs = 0

    # 分批处理
    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Mining Batches"):
        batch_prompts = prompts[i : i + BATCH_SIZE]

        # 1. 执行推理
        outputs = llm.generate(batch_prompts, sampling_params)

        batch_data = []

        # 2. 解析 JSON
        for output in outputs:
            gen_text = output.outputs[0].text
            try:
                # 提取 JSON 数组
                json_match = re.search(r'\[.*\]', gen_text, re.DOTALL)
                if json_match:
                    pairs = json.loads(json_match.group())
                    for p in pairs:
                        if 'akk' in p and 'en' in p:
                            # 简单的长度过滤
                            if len(p['akk']) > 5 and len(p['en']) > 5:
                                batch_data.append({
                                    'transliteration': p['akk'],
                                    'translation': p['en']
                                })
            except:
                continue

        # 3. 写入文件 (Append Mode)
        if batch_data:
            df_batch = pd.DataFrame(batch_data)
            df_batch.to_csv(RAW_OUTPUT_CSV, mode='a', header=False, index=False)
            total_pairs += len(batch_data)

            # 4. 实时质量反馈 (看一眼刚才挖到了什么)
            print(f"\n✨ Batch {i//BATCH_SIZE + 1}: Found {len(batch_data)} pairs.")
            print(f"   Sample: {batch_data[0]['transliteration']} -> {batch_data[0]['translation']}")
        else:
            print(f"\n⚠️ Batch {i//BATCH_SIZE + 1}: No pairs found (empty or failed parse).")

    print(f"\n🎉 Mining Complete! Total raw pairs: {total_pairs}")

# ==========================================
# 6. 最终清洗与标准化 (Final Cleaning)
# ==========================================
def clean_and_finalize():
    print("🛁 [Step 4] Cleaning and formatting final dataset...")
    if not os.path.exists(RAW_OUTPUT_CSV):
        print("No mined data found to clean.")
        return

    df = pd.read_csv(RAW_OUTPUT_CSV)
    print(f"   Raw loaded: {len(df)}")

    # 去重
    df = df.drop_duplicates(subset=['transliteration'])

    # 定义清洗函数 (与训练代码一致)
    def replace_gaps(text):
        text = str(text)
        text = re.sub(r'\.3(?:\s+\.3)+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
        text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)
        text = re.sub(r'xx', '<gap>', text)
        text = re.sub(r' x ', ' <gap> ', text)
        text = re.sub(r'…', '<big_gap>', text)
        return text

    def basic_normalize_target(s):
        s = str(s)
        s = re.sub(r"[ \t]+", " ", s)
        s = re.sub(r"\s+([,.;:!?])", r"\1", s)
        s = re.sub(r"([,.;:!?])([A-Za-z])", r"\1 \2", s)
        return s.strip()

    # 应用清洗
    df['transliteration'] = df['transliteration'].apply(replace_gaps)
    df['translation'] = df['translation'].apply(basic_normalize_target)

    # 再次过滤太短的
    df = df[df['transliteration'].str.len() > 3]
    df = df[df['translation'].str.len() > 3]

    # 保存
    df.to_csv(FINAL_OUTPUT_CSV, index=False)
    print(f"✅ FINAL DATASET SAVED: {FINAL_OUTPUT_CSV}")
    print(f"   Final Count: {len(df)} pairs")
    print("   (You can now load this CSV in your training notebook!)")

# ==========================================
# 执行入口
# ==========================================
if __name__ == "__main__":
    # 1. 粗筛
    candidates = coarse_filter(INPUT_CSV)

    if len(candidates) > 0:
        # 2. 准备 Prompt
        prompts = construct_prompts(candidates)

        # 3. 初始化模型
        llm, sampling_params = init_llm()

        # 4. 开始挖掘
        run_mining(llm, sampling_params, prompts)

        # 5. 清洗打包
        clean_and_finalize()
    else:
        print("❌ No pages passed coarse filtering.")

In [ ]:
#清洗脚本用于清洗前面提取的csv

import pandas as pd
import numpy as np
import re
import json
import torch
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

# ==========================================
# 1. 配置区域
# ==========================================
FILES = ["mined_raw_h100 (1).csv", "mined_raw_h100 (2).csv", "mined_raw_h100 (4).csv"]
INTERMEDIATE_CSV = "washed_candidates.csv"
FINAL_SCORED_CSV = "mined_scored_final.csv"

# LLM 配置 (H100 80G 推荐 72B-Int4)
MODEL_PATH = "Qwen/Qwen2.5-72B-Instruct-GPTQ-Int4"

# ==========================================
# 2. 阶段一：启发式粗筛 (The Heuristic Wash)
# ==========================================
def heuristic_wash(file_list):
    print("🧹 [Stage 1] Starting Heuristic Wash...")
    dfs = []
    for f in file_list:
        dfs.append(pd.read_csv(f))
    df = pd.concat(dfs, ignore_index=True)

    # a. 去重
    df = df.drop_duplicates(subset=['transliteration'])

    # b. 长度比过滤 (0.5 - 1.5)
    df['src_len'] = df['transliteration'].astype(str).str.len()
    df['tgt_len'] = df['translation'].astype(str).str.len()
    df['ratio'] = df['src_len'] / df['tgt_len']
    df = df[(df['ratio'] >= 0.5) & (df['ratio'] <= 1.5)]

    # c. OCR 乱码/特殊符号过滤 (剔除包含 ˜, †, ª, º 等字符)
    bad_chars = r'[˜†ªº\^\|/\[\]]'
    df = df[~df['translation'].str.contains(bad_chars, na=False)]
    df = df[~df['transliteration'].str.contains(bad_chars, na=False)]

    # d. 德语/法语黑名单
    german_stops = r'\b(und|der|die|das|von|hat|ist|mit|auf|für|den|dem|dans|pour)\b'
    df = df[~df['translation'].str.contains(german_stops, case=False, regex=True, na=False)]

    # e. 学术废话过滤
    academic_stops = r'\b(Page|Vol|Plate|Fig|ibid|No\.)\b'
    df = df[~df['translation'].str.contains(academic_stops, case=False, regex=True, na=False)]

    print(f"   -> Wash complete. Remaining candidates: {len(df)}")
    return df.reset_index(drop=True)

# ==========================================
# 3. 阶段二：LLM 打分 (LLM-as-a-Judge)
# ==========================================
def run_llm_scoring(df):
    print(f"🚀 [Stage 2] Loading {MODEL_PATH} for expert scoring...")

    llm = LLM(
        model=MODEL_PATH,
        gpu_memory_utilization=0.9,
        max_model_len=4096,
        trust_remote_code=True,
        enforce_eager=True
    )

    # 打分不需要随机性，使用贪婪采样
    sampling_params = SamplingParams(temperature=0.0, max_tokens=150)

    system_prompt = """You are an expert Assyriologist. Your task is to rate the quality of an Akkadian-to-English translation pair.
Score 0 to 10 based on:
1. Accuracy: Does the English accurately translate the Akkadian transliteration?
2. Noise: Is it free from OCR errors, German/French words, or academic citations?
3. Grammar: Is it a complete, meaningful sentence or phrase (not a random fragment)?

Output format: JSON ONLY like {"score": 8.5, "reason": "..."}"""

    prompts = []
    for _, row in df.iterrows():
        p = f"""<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
Rate this pair:
Akk: {row['transliteration']}
En: {row['translation']}<|im_end|>
<|im_start|>assistant
{{"score": """  # 引导模型直接输出分数
        prompts.append(p)

    print(f"   -> Running inference on {len(prompts)} candidates...")
    outputs = llm.generate(prompts, sampling_params)

    scores = []
    reasons = []

    for output in outputs:
        # 补全 JSON 并解析
        res_text = '{"score": ' + output.outputs[0].text
        try:
            # 兼容处理可能的多余文本
            json_match = re.search(r'\{.*\}', res_text, re.DOTALL)
            data = json.loads(json_match.group())
            scores.append(float(data.get('score', 0)))
            reasons.append(data.get('reason', ""))
        except:
            scores.append(0.0)
            reasons.append("Parse error")

    df['llm_score'] = scores
    df['llm_reason'] = reasons
    return df

# ==========================================
# 4. 执行流程
# ==========================================
if __name__ == "__main__":
    # 第一步：粗筛
    df_washed = heuristic_wash(FILES)
    df_washed.to_csv(INTERMEDIATE_CSV, index=False)

    # 第二步：打分 (如果数据量很大，建议先拿前 1000 条测试)
    # df_test = df_washed.head(1000)
    df_final = run_llm_scoring(df_washed)

    # 第三步：保存结果
    df_final.to_csv(FINAL_SCORED_CSV, index=False)

    # 打印一些高质量样本看看
    print("\n🏆 Top Quality Mined Data (Score > 9.0):")
    print(df_final[df_final['llm_score'] >= 9.0][['transliteration', 'translation', 'llm_score']].head(10))

In [ ]:
# #手工拆分长段落prompt
# # Role
# 你是一位精通古代近东语言（阿卡德语/亚述语）的语言学专家，擅长将转写文本（Transliteration）与英文译文（Translation）进行精确的句子级对齐。
# # Task
# 将提供的转写原文段落与译文段落切分为多个对齐的片段（Segments），采用最粗粒度切分因为切多错多。
# # Constraints & Rules
# 1. **长度限制**：每个片段的原文和译文长度均必须严格小于 512 字符。
# 2. **语义锚点对齐（核心）**：严禁仅按字数比例切分。必须通过“硬锚点”确保原文与译文的同步。优先匹配：
#    - **数值**：如 "92", "27.83333", "100"。
#    - **度量衡单位**：如 "ma-na" (minas), "GÍN" (shekels)。
#    - **关键名词/动词**：如 "ANŠE" (donkeys), "um-ma...-ma" (saying:), "qí-bi-ma" (speak to)。
# 3. **去噪逻辑**：如果原文（Transliteration）因泥板损坏或录入缺失而提前结束，必须丢弃译文中多余的、无对应原文的内容。不提供“孤儿译文”。
# 4. **片段完整性**：在不超出长度限制的前提下，尽量保持一个交易逻辑（如：某项货物的数量、单价和总价）在同一个片段内。

In [ ]:
#extra数据清洗prompt
# ## System Prompt

# ```
# You are cleaning a parallel corpus of Akkadian transliterations and English translations.

# Apply ONLY the following rules. Do NOT normalize, retranslate, or restructure anything beyond what is specified.

# ## INPUT
# Each row: `transliteration` | `translation`

# ## OUTPUT
# JSON array. Each entry:
# - {"action": "keep", "transliteration": "...", "translation": "..."}
# - {"action": "drop", "reason": "..."}

# ## RULE 1: Remove leaked line numbers from transliteration

# Akkadian transliterations consist of hyphenated syllable sequences (e.g., a-na, qi-bi4-ma, É.GAL-lim) and Sumerograms (e.g., KÙ.BABBAR, TÚG.HI.A). Standalone integers or alphanumeric fragments that appear as editorial line numbering must be removed.

# REMOVE these patterns:
# - A standalone integer between two hyphenated sign groups:
#     "la tù-lâ-bi4-is 16 Sé-sû-ur" → "la tù-lâ-bi4-is Sé-sû-ur"
# - A standalone integer before a hyphenated sign group at the start of the text:
#     "1 a-na Pu-šu-ke-en6 qí-bi4-ma 2 um-ma Dan-A-šùr-ma" → "a-na Pu-šu-ke-en6 qí-bi4-ma um-ma Dan-A-šùr-ma"
# - OCR-garbled line numbers at the start (letter+digit fragments like "z1"):
#     "z1 a-na Wa-wa-li" → "a-na Wa-wa-li"
# - A standalone integer immediately before a Sumerogram:
#     "14 SÀ.BA 1 ma-na" → "SÀ.BA 1 ma-na" (remove 14, keep 1 because "1 ma-na" is a quantity)

# DO NOT REMOVE integers that are:
# - Part of sign readings with subscript digits: bi4, il5, de8, en6, etc.
# - Quantities followed by a unit or measure: "10 GÚ", "5 GÍN", "50 TÚG.HI.A", "1 ma-na", "30 GÚ", "36 TÚG"
# - Part of date expressions: "ITU.6.KAM", "MU 1"

# KEY DISTINCTION: A line number is a bare integer whose removal does not break the meaning of the surrounding text. A quantity is an integer directly modifying a following unit/noun. When in doubt, check if the translation contains a corresponding quantity — if it does, keep the number; if not, it is a line number.

# ## RULE 2: Remove leaked line numbers and editorial markers from translation

# Translation text may contain leaked line-number references. Remove them.

# REMOVE:
# - Standalone integers used as interlinear cross-references embedded in running English:
#     "You always have been 1 a great help 2 to me; 4 may 3 Marduk 4 bless you" → "You always have been a great help to me; may Marduk bless you"
# - Line-range markers before clauses: "3b-9", "14b-18a", "rev."
# - Superscript-style letter markers: "a", "b" appended to words when clearly editorial (e.g., "Tari a." → "Tari." only if clearly a stray marker)
# - German or other non-English editorial text: "Betrifft", "Die Lesung ... ist gesichert durch..."
# - Cross-references: "from line 33"
# - Philological footnotes about readings

# DO NOT REMOVE:
# - Numbers that are part of the translated content: "10 years", "5 days", "36 textiles", "6 shekels"
# - Technical Akkadian terms: sikkātum, nishātu, kaššum, etc.
# - <gap> markers

# KEY DISTINCTION: If removing an integer breaks the English sentence's grammar or meaning, it is content — keep it. If the sentence reads more naturally without the integer, it is a line number — remove it.

# ## RULE 3: Drop duplicates
# If two rows overlap >90% in transliteration, keep the cleaner version, drop the other with reason "duplicate".

# ## RULE 4: Drop irrecoverable rows
# Drop a row if its transliteration is majority corrupted:
# - Non-Akkadian gibberish (°, ü, random Latin fragments)
# - Sequences clearly garbled beyond recognition (wa-cxs-ba-ku-ni, üntammm-am, tva-cts-bu)
# - Encoding artifacts dominating the line

# Threshold: If >60% of signs are recognizable Akkadian syllables or Sumerograms, KEEP. If <60%, DROP.

# ## DO NOT TOUCH
# - OCR variants that still look like plausible Akkadian readings
# - Sumerogram casing
# - <gap> markers
# - Translation wording, phrasing, style beyond line-number removal
# - Technical terms, proper nouns
# - Anything you are unsure about
# ```

# ## User Prompt

# ```
# Clean the following rows according to ONLY the 4 rules in your instructions. Return JSON.

# ---
# {paste rows}
# ---
# ```

In [ ]:
# #ocr石板数据提取prompt
# You are given an Old Assyrian letter with parallel transliteration and  English translation.  ## Input Format - The header contains a text ID in the format "Kt XX/k NNN" (e.g.,    "Kt 92/k 209"). Extract this as the unique document ID. - Line-numbered Akkadian transliteration on the left, English    translation on the right. - Some lines are marked with "e." (edge) or "rev." (reverse). ## Task 1. Extract the document ID (e.g., "Kt 92/k 209") from the header. 2. Segment the text into sentence pairs aligned by periods (.)     in the English translation. 3. For each English sentence, find the corresponding Akkadian lines     by matching line numbers. 4. Concatenate multi-line Akkadian segments with spaces. 5. Preserve all diacritics, subscript numbers, and special characters     (e.g., ₂, š, ù, á) exactly. 6. Ignore editorial annotations in square brackets [ ]. ## Output Format JSON array: ```json

#模型训练

In [ ]:
#orig和extra数据：https://drive.google.com/file/d/1ziYQjxETbT8t-UXLK2IxU9IszAv6vOJh/view?usp=drive_link
#sliver数据：https://drive.google.com/file/d/1qMsbeuLJBxqbBhcpjLytD5n7GWXhq8tp/view?usp=sharing
#unlabel数据（伪标签）：https://drive.google.com/file/d/1n4OOVy9PzMHh96gMe9oJmQuHJ3x3TflD/view?usp=sharing

预处理格式规范

In [ ]:
#预处理
import pandas as pd
import re

def v3_official_clean_source(text):
    """
    原文 (Transliteration) 严格遵循 v3 官方公告的清洗
    """
    if not isinstance(text, str): return ""

    # 1. 明确的 Gap 替换
    text = re.sub(r'\[x\]|\(break\)|\(large break\)|\(\d+ broken lines\)', '<gap>', text)
    text = re.sub(r'\b[xX]\b|\.\.\.', '<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    while '<gap> <gap>' in text:
        text = text.replace('<gap> <gap>', '<gap>')

    # 2. 明确的 Optional changes
    text = text.replace('Ḫ', 'H').replace('ḫ', 'h')
    text = text.replace('KÙ.B.', 'KÙ.BABBAR')

    # 3. 明确的 Unicode subscript numbers
    sub_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    text = text.translate(sub_map)

    # 4. 明确的 Decimals to Fractions
    fractions_map = {
        '0.5': '½', '0.25': '¼', '0.3333': '⅓', '0.8333': '⅚',
        '0.625': '⅝', '0.6666': '⅔', '0.75': '¾', '0.1666': '⅙'
    }
    for dec, frac in fractions_map.items():
        text = text.replace(dec, frac)

    return text.strip()


def v3_official_clean_target(text):
    """
    译文 (Translation) 严格遵循 v3 官方公告 + 句式标准化
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = text.strip()

    # 1. 明确的 "Remove from translations"
    text = text.replace('fem.', '')
    text = text.replace('sing.', '')
    text = text.replace('pl.', '')
    text = text.replace('plural', '')
    text = text.replace('(?)', '')
    text = text.replace('..', '')
    text = text.replace('?', '') # 去除 stray ?
    text = re.sub(r'\bxx+\b|\bx\b', '', text)
    text = text.replace('<<', '').replace('>>', '')
    text = re.sub(r'<(?!gap>)[^>]*>', '', text)

    # 2. 明确的 Gap 替换
    text = re.sub(r'\[x\]|\(break\)|\(large break\)|\(\d+ broken lines\)', '<gap>', text)
    text = re.sub(r'\b[xX]\b|\.\.\.', '<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    while '<gap> <gap>' in text:
        text = text.replace('<gap> <gap>', '<gap>')

    # 3. 明确的 "Replace in translations"
    text = re.sub(r'\bPN\b', '<gap>', text)
    text = text.replace('-gold', ' pašallum gold')
    text = text.replace('-tax', ' šadduātum tax')
    text = text.replace('-textiles', ' kutānum textiles')

    # 4. 明确的复杂重量与分数转换
    weight_replacements = {
        '1 / 12 (shekel)': '15 grains',
        '5 / 12 shekel': '⅓ shekel 15 grains',
        '5 11 / 12 shekels': '6 shekels less 15 grains',
        '7 / 12 shekel': '½ shekel 15 grains'
    }
    for old_w, new_w in weight_replacements.items():
        text = text.replace(old_w, new_w)

    fractions_map = {
        '0.5': '½', '0.25': '¼', '0.3333': '⅓', '0.8333': '⅚',
        '0.625': '⅝', '0.6666': '⅔', '0.75': '¾', '0.1666': '⅙'
    }
    for dec, frac in fractions_map.items():
        text = text.replace(dec, frac)

    # 5. 明确的 "Roman numerals to integer numbers for months"
    months_map = {
        'Month I': 'Month 1', 'Month II': 'Month 2', 'Month III': 'Month 3',
        'Month IV': 'Month 4', 'Month V': 'Month 5', 'Month VI': 'Month 6',
        'Month VII': 'Month 7', 'Month VIII': 'Month 8', 'Month IX': 'Month 9',
        'Month X': 'Month 10', 'Month XI': 'Month 11', 'Month XII': 'Month 12'
    }
    for rom, arab in months_map.items():
        text = text.replace(rom, arab)

    # 6. 基础格式化清理
    text = re.sub(r'\s+', ' ', text).strip()
    if not text: return ""

    # ==========================================
    # 7. 官方认可的句式标准化 (你的首字母与句号规则)
    # ==========================================
    # 强制首字母大写 (如果第一个字符是字母)
    if text[0].islower():
        text = text[0].upper() + text[1:]

    # 强制结尾补齐句号 (规避 extra 数据的截断感，迎合官方评估风格)
    # 如果结尾不是标准句末标点或 gap 标签，则补上句号
    if text[-1] not in ['.', '!', '?', '"', '\'', '>']:
        text += '.'

    return text

def apply_minimalist_cleaning(input_file, output_file):
    df = pd.read_csv(input_file)

    if 'transliteration' in df.columns:
        df['transliteration'] = df['transliteration'].apply(v3_official_clean_source)
    if 'translation' in df.columns:
        df['translation'] = df['translation'].apply(v3_official_clean_target)

    initial_len = len(df)
    df = df.drop_duplicates().reset_index(drop=True)

    df.to_csv(output_file, index=False)
    print(f"处理完成！{input_file} -> {output_file}")
    print(f"原始行数: {initial_len}, 去重后行数: {len(df)}")

# --- 执行清洗 ---
apply_minimalist_cleaning('orig.csv', 'orig_min_cleaned.csv')
apply_minimalist_cleaning('extra.csv', 'extra_min_cleaned.csv')

# byt5（trian and inference）

In [ ]:
#数据清洗提升3+
#格式规范提升0.2
#presudo label 提升 0.6
#data augument 提升0.4
#更大的模型提升2+
#lb = 37


# 接下来提分点
# 第一优先级：更规范的数据格式处理（预处理后处理），esamble
# 第二优先级：数据增强
# 第三优先级：超参数精调

!pip install evaluate sacrebleu
import os, gc, re, math, random
import pandas as pd
import numpy as np
import torch
from collections import defaultdict
from sklearn.model_selection import train_test_split
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer
)
import evaluate

# ==========================================
# 1. Config & Seed
# ==========================================
class Config:
    MODEL_NAME = "google/byt5-large"
    MAX_LENGTH = 512
    BATCH_SIZE = 8
    EPOCHS = 6
    LEARNING_RATE = 1.5e-4
    OUTPUT_DIR = "./byt5-akkadian-model"

def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    random.seed(seed)

seed_everything()

# ==========================================
# 2. Data Loading & Sentence Alignment & Augmentation
# ==========================================
INPUT_DIR = "/kaggle/input/akkadian2english-dataset"
EXTRA_DATA = "/kaggle/input/datasets/minminandy/new-format-cleaned/extra_min_cleaned (1).csv"
LEXICON_DATA = "/kaggle/input/akkadian2english-dataset/OA_Lexicon_eBL.csv"
PSEUDO_DATA = "/kaggle/input/datasets/minminandy/pseudotest/pseudo_top_filtered.csv"

train_df = pd.read_csv("/kaggle/input/datasets/minminandy/new-format-cleaned/orig_min_cleaned (1).csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")
extra_df = pd.read_csv(EXTRA_DATA)
lexicon_df = pd.read_csv(LEXICON_DATA)
pseudo_df = pd.read_csv(PSEUDO_DATA)

print(f"Original Train Data: {len(train_df)} docs")
print(f"Extra Mined Data: {len(extra_df)} sentences")
print(f"Lexicon Data: {len(lexicon_df)} entries")
print(f"Pseudo Label Data: {len(pseudo_df)} sentences")

def simple_sentence_aligner(df):
    aligned_data = []
    for idx, row in df.iterrows():
        src = str(row['transliteration'])
        tgt = str(row['translation'])
        tgt_sents = [t.strip() for t in re.split(r'(?<=[.!?])\s+', tgt) if t.strip()]
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:
                    aligned_data.append({'transliteration': s, 'translation': t})
        else:
            aligned_data.append({'transliteration': src, 'translation': tgt})
    return pd.DataFrame(aligned_data)

def get_augmentation_data(df, lexicon):
    BLACK_LIST = {'a-na', 'i-na', 'ma-na', 'šu-ma', 'a-ta', 'um-ma', 'ù', 'ša'}
    pn_gn = lexicon[lexicon['type'].isin(['PN', 'GN'])].copy()

    lexeme_to_forms = defaultdict(set)
    form_to_lexeme = {}
    for _, row in pn_gn.iterrows():
        f, l = str(row['form']), str(row['lexeme'])
        if f not in BLACK_LIST and len(f) > 3:
            lexeme_to_forms[l].add(f)
            form_to_lexeme[f] = l

    def augment_line(text):
        tokens = text.split()
        new_tokens = []
        changed = False
        for token in tokens:
            clean = token.strip(',.;:!?()[]')
            punc_prefix = token[:token.find(clean)] if clean in token else ""
            punc_suffix = token[token.find(clean) + len(clean):] if clean in token else ""
            if clean in form_to_lexeme:
                others = [v for v in lexeme_to_forms[form_to_lexeme[clean]] if v != clean]
                if others:
                    new_tokens.append(punc_prefix + random.choice(others) + punc_suffix)
                    changed = True
                    continue
            new_tokens.append(token)
        return " ".join(new_tokens), changed

    aug_results = []
    for _, row in df.iterrows():
        aug_text, is_changed = augment_line(str(row['transliteration']))
        if is_changed:
            aug_results.append({
                'transliteration': aug_text,
                'translation': row['translation']
            })
    return pd.DataFrame(aug_results)

# --- 执行句子对齐 ---
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences")

# --- 清洗额外数据 ---
extra_clean = extra_df[['transliteration', 'translation']].dropna().copy()
extra_clean = extra_clean[
    (extra_clean['transliteration'].str.len() > 3) &
    (extra_clean['translation'].str.len() > 3)]
print(f"Extra Data (after cleaning): {len(extra_clean)} sentences")

# --- 清洗 pseudo label 数据 ---
pseudo_clean = pseudo_df[['transliteration', 'translation']].dropna().copy()
pseudo_clean = pseudo_clean[
    (pseudo_clean['transliteration'].str.len() > 3) &
    (pseudo_clean['translation'].str.len() > 3)]
print(f"Pseudo Label Data (after cleaning): {len(pseudo_clean)} sentences")

# --- 先 split，再增强（避免 val 泄露） ---
dataset_orig = Dataset.from_pandas(train_expanded)
split_datasets = dataset_orig.train_test_split(test_size=0.1, seed=42)

# 只对训练集做增强
train_only_df = split_datasets["train"].to_pandas()
aug_df = get_augmentation_data(train_only_df, lexicon_df)
print(f"Lexicon Augmented Data: {len(aug_df)} sentences")

# 将额外数据 + 增强数据 + pseudo label 合并到训练集
extra_dataset = Dataset.from_pandas(extra_clean.reset_index(drop=True))
aug_dataset = Dataset.from_pandas(aug_df.reset_index(drop=True))
pseudo_dataset = Dataset.from_pandas(pseudo_clean.reset_index(drop=True))

split_datasets["train"] = concatenate_datasets([
    split_datasets["train"],
    extra_dataset,
    aug_dataset,
    pseudo_dataset,
])
print(f"Final Train: {len(split_datasets['train'])} | Val: {len(split_datasets['test'])}")

# ==========================================
# 3. Tokenization
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
PREFIX = "translate Akkadian to English: "

def preprocess_function(examples):
    inputs = [PREFIX + str(ex) for ex in examples["transliteration"]]
    targets = [str(ex) for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=Config.MAX_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = split_datasets["train"].map(preprocess_function, batched=True)
tokenized_val = split_datasets["test"].map(preprocess_function, batched=True)

# ==========================================
# 4. Training
# ==========================================
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]

    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_wrapped = [[l.strip()] for l in decoded_labels]

    bleu_score = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels_wrapped)["score"]
    chrf_score = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels_wrapped, word_order=2)["score"]
    geo_mean = math.sqrt(bleu_score * chrf_score)

    return {"bleu": bleu_score, "chrf++": chrf_score, "geo_mean": geo_mean}

args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=Config.LEARNING_RATE,
    fp16=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    save_only_model=True,
    save_safetensors=True,
    num_train_epochs=Config.EPOCHS,
    predict_with_generate=True,
    generation_max_length=Config.MAX_LENGTH,
    generation_num_beams=2,
    logging_steps=10,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting Training (FP32 mode)...")
trainer.train()

# ==========================================
# 5. Save Model
# ==========================================
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")

In [ ]:
import os
import pandas as pd
import torch
import re
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

# ==========================================
# Configuration
# ==========================================
MODEL_PATH = "/kaggle/input/notebooks/zhuyifanss/biggestbyt5/byt5-akkadian-model/"
TEST_DATA_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
BATCH_SIZE = 8
MAX_LENGTH = 512
NUM_BEAMS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREFIX = "translate Akkadian to English: "

# ==========================================
# 严格对齐训练集 v3_official 预处理
# ==========================================

def v3_official_clean_source(text):
    if not isinstance(text, str): return ""

    # 1. Gap 替换
    text = re.sub(r'\[x\]|\(break\)|\(large break\)|\(\d+ broken lines\)', '<gap>', text)
    text = re.sub(r'\b[xX]\b|\.\.\.', '<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    while '<gap> <gap>' in text:
        text = text.replace('<gap> <gap>', '<gap>')

    # 2. Optional changes
    text = text.replace('Ḫ', 'H').replace('ḫ', 'h')
    text = text.replace('KÙ.B.', 'KÙ.BABBAR')

    # 3. Unicode subscript numbers
    sub_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    text = text.translate(sub_map)

    # 4. Decimals to Fractions
    fractions_map = {
        '0.5': '½', '0.25': '¼', '0.3333': '⅓', '0.8333': '⅚',
        '0.625': '⅝', '0.6666': '⅔', '0.75': '¾', '0.1666': '⅙'
    }
    for dec, frac in fractions_map.items():
        text = text.replace(dec, frac)

    return text.strip()


def v3_official_clean_target(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = text.strip()

    # 1. Remove from translations
    text = text.replace('fem.', '')
    text = text.replace('sing.', '')
    text = text.replace('pl.', '')
    text = text.replace('plural', '')
    text = text.replace('(?)', '')
    text = text.replace('..', '')
    text = text.replace('?', '')
    text = re.sub(r'\bxx+\b|\bx\b', '', text)
    text = text.replace('<<', '').replace('>>', '')
    text = re.sub(r'<(?!gap>)[^>]*>', '', text)

    # 2. Gap 替换
    text = re.sub(r'\[x\]|\(break\)|\(large break\)|\(\d+ broken lines\)', '<gap>', text)
    text = re.sub(r'\b[xX]\b|\.\.\.', '<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    while '<gap> <gap>' in text:
        text = text.replace('<gap> <gap>', '<gap>')

    # 3. Replace in translations
    text = re.sub(r'\bPN\b', '<gap>', text)
    text = text.replace('-gold', ' pašallum gold')
    text = text.replace('-tax', ' šadduātum tax')
    text = text.replace('-textiles', ' kutānum textiles')

    # 4. 复杂重量与分数转换
    weight_replacements = {
        '1 / 12 (shekel)': '15 grains',
        '5 / 12 shekel': '⅓ shekel 15 grains',
        '5 11 / 12 shekels': '6 shekels less 15 grains',
        '7 / 12 shekel': '½ shekel 15 grains'
    }
    for old_w, new_w in weight_replacements.items():
        text = text.replace(old_w, new_w)

    fractions_map = {
        '0.5': '½', '0.25': '¼', '0.3333': '⅓', '0.8333': '⅚',
        '0.625': '⅝', '0.6666': '⅔', '0.75': '¾', '0.1666': '⅙'
    }
    for dec, frac in fractions_map.items():
        text = text.replace(dec, frac)

    # 5. Roman numerals to integer numbers for months
    months_map = {
        'Month I': 'Month 1', 'Month II': 'Month 2', 'Month III': 'Month 3',
        'Month IV': 'Month 4', 'Month V': 'Month 5', 'Month VI': 'Month 6',
        'Month VII': 'Month 7', 'Month VIII': 'Month 8', 'Month IX': 'Month 9',
        'Month X': 'Month 10', 'Month XI': 'Month 11', 'Month XII': 'Month 12'
    }
    for rom, arab in months_map.items():
        text = text.replace(rom, arab)

    # 6. 基础格式化清理
    text = re.sub(r'\s+', ' ', text).strip()
    if not text: return ""

    # 7. 句式标准化
    if text[0].islower():
        text = text[0].upper() + text[1:]

    if text[-1] not in ['.', '!', '?', '"', '\'', '>']:
        text += '.'

    return text

# ==========================================
# Model Loading
# ==========================================
print(f"Loading model from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()

# ==========================================
# Data Preparation
# ==========================================
test_df = pd.read_csv(TEST_DATA_PATH)
test_df['clean_transliteration'] = test_df['transliteration'].apply(v3_official_clean_source)
print(f"Test samples: {len(test_df)}")

class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = [PREFIX + str(t) for t in texts]
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
        }

test_dataset = InferenceDataset(test_df['clean_transliteration'].tolist(), tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# Inference
# ==========================================
print("Starting Inference...")
all_predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend(decoded)

# ==========================================
# Submission
# ==========================================
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": all_predictions,
})

submission["translation"] = submission["translation"].apply(v3_official_clean_target)

submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully!")
print(submission.head())

# qwen8b

In [ ]:
# ============================================================
# Qwen 3 8B Akkadian Translation - Training Pipeline (Full Data)
# lb = 36.5
# ============================================================

# --- 0. 环境安装 ---
try:
    import unsloth
except Exception:
    get_ipython().system('pip uninstall -y torchvision torch')
    get_ipython().system('pip install unsloth==2025.11.6')
    get_ipython().system('pip uninstall -y tensorflow')

get_ipython().system('pip install sacrebleu -q')

import os, gc, re, random, math
import pandas as pd
import numpy as np
import torch
import sacrebleu
from collections import defaultdict
from sklearn.model_selection import train_test_split
from datasets import Dataset, concatenate_datasets
from unsloth import FastLanguageModel, is_bfloat16_supported

# ============================================================
# 1. 配置
# ============================================================
class Config:
    MODEL_NAME = "unsloth/Qwen3-8B"
    LOAD_IN_4BIT = False
    LORA_RANK = 32
    LORA_ALPHA = 64
    LORA_DROPOUT = 0.1
    MAX_SEQ_LENGTH = 512

    EPOCHS = 3
    BATCH_SIZE = 16
    GRAD_ACCUM = 1
    LR = 2e-4
    WARMUP_STEPS = 10
    WEIGHT_DECAY = 0.05
    SEED = 42
    OUTPUT_DIR = "./qwen3-akkadian-lora"

    DATA_DIR = "/kaggle/input/akkadian2english-dataset"
    ORIG_TRAIN = "/kaggle/input/datasets/minminandy/new-format-cleaned/orig_min_cleaned (1).csv"
    EXTRA_DATA = "/kaggle/input/datasets/minminandy/new-format-cleaned/extra_min_cleaned (1).csv"
    EXTRA2_DATA = "/kaggle/input/datasets/minminandy/pseudo-extra2/pseudo_dataset_cleaned.csv"
    PSEUDO_DATA = "/kaggle/input/datasets/minminandy/pseudotest/pseudo_top_filtered.csv"

def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)

seed_everything(Config.SEED)

# ============================================================
# 2. 数据加载 & 句子对齐 & 增强
# ============================================================
train_df = pd.read_csv(Config.ORIG_TRAIN)
test_df = pd.read_csv(f"{Config.DATA_DIR}/test.csv")
lexicon_df = pd.read_csv(f"{Config.DATA_DIR}/OA_Lexicon_eBL.csv")

print(f"原始训练数据: {len(train_df)} docs")

def simple_sentence_aligner(df):
    aligned_data = []
    for idx, row in df.iterrows():
        src = str(row['transliteration'])
        tgt = str(row['translation'])
        tgt_sents = [t.strip() for t in re.split(r'(?<=[.!?])\s+', tgt) if t.strip()]
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:
                    aligned_data.append({'transliteration': s, 'translation': t})
        else:
            aligned_data.append({'transliteration': src, 'translation': tgt})
    return pd.DataFrame(aligned_data)

train_expanded = simple_sentence_aligner(train_df)
print(f"句子对齐后: {len(train_expanded)} sentences")

extra_clean = None
if os.path.exists(Config.EXTRA_DATA):
    extra_df = pd.read_csv(Config.EXTRA_DATA)
    extra_clean = extra_df[['transliteration', 'translation']].dropna()
    extra_clean = extra_clean[
        (extra_clean['transliteration'].str.len() > 3) &
        (extra_clean['translation'].str.len() > 3)
    ]
    print(f"额外数据 (清洗后): {len(extra_clean)} 条")
else:
    print(f"警告: 文件不存在 {Config.EXTRA_DATA}")

extra2_clean = None
if os.path.exists(Config.EXTRA2_DATA):
    extra2_df = pd.read_csv(Config.EXTRA2_DATA)
    extra2_clean = extra2_df[['transliteration', 'translation']].dropna()
    extra2_clean = extra2_clean[
        (extra2_clean['transliteration'].str.len() > 3) &
        (extra2_clean['translation'].str.len() > 3)
    ]
    print(f"额外数据2 (清洗后): {len(extra2_clean)} 条")
else:
    print(f"警告: 文件不存在 {Config.EXTRA2_DATA}")

pseudo_clean = None
if os.path.exists(Config.PSEUDO_DATA):
    pseudo_df = pd.read_csv(Config.PSEUDO_DATA)
    pseudo_clean = pseudo_df[['transliteration', 'translation']].dropna().copy()
    pseudo_clean = pseudo_clean[
        (pseudo_clean['transliteration'].str.len() > 3) &
        (pseudo_clean['translation'].str.len() > 3)
    ]
    print(f"Pseudo Label 数据 (清洗后): {len(pseudo_clean)} 条")
else:
    print(f"警告: Pseudo label 文件不存在 {Config.PSEUDO_DATA}")

def get_augmentation_data(df, lexicon):
    BLACK_LIST = {'a-na', 'i-na', 'ma-na', 'šu-ma', 'a-ta', 'um-ma', 'ù', 'ša'}
    pn_gn = lexicon[lexicon['type'].isin(['PN', 'GN'])].copy()
    lexeme_to_forms = defaultdict(set)
    form_to_lexeme = {}
    for _, row in pn_gn.iterrows():
        f, l = str(row.get('form', '')), str(row.get('lexeme', ''))
        if f not in BLACK_LIST and len(f) > 3:
            lexeme_to_forms[l].add(f)
            form_to_lexeme[f] = l
    aug_results = []
    for _, row in df.iterrows():
        tokens = str(row['transliteration']).split()
        new_tokens, changed = [], False
        for token in tokens:
            clean = token.strip(',.;:!?()[]')
            if clean in form_to_lexeme:
                others = [v for v in lexeme_to_forms[form_to_lexeme[clean]] if v != clean]
                if others:
                    punc_pre = token[:token.find(clean)] if clean in token else ""
                    punc_suf = token[token.find(clean) + len(clean):] if clean in token else ""
                    new_tokens.append(punc_pre + random.choice(others) + punc_suf)
                    changed = True
                    continue
            new_tokens.append(token)
        if changed:
            aug_results.append({
                'transliteration': " ".join(new_tokens),
                'translation': row['translation']
            })
    return pd.DataFrame(aug_results)

train_split, val_split = train_test_split(
    train_expanded, test_size=0.1, random_state=Config.SEED
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

aug_df = get_augmentation_data(train_split, lexicon_df)
print(f"词典增强: {len(aug_df)} 条")

all_parts = [train_split, aug_df]
if extra_clean is not None:
    all_parts.append(extra_clean)
if extra2_clean is not None:
    all_parts.append(extra2_clean)
if pseudo_clean is not None:
    all_parts.append(pseudo_clean)

all_train = pd.concat(all_parts, ignore_index=True)
print(f"\n最终训练集: {len(all_train)} | 验证集: {len(val_split)}")

# ============================================================
# 3. 构建 Chat 格式数据集
# ============================================================
SYSTEM_PROMPT = "You are a professional Akkadian translation expert. Translate the given Akkadian transliteration into English accurately."

def create_conversations(df, is_train=True):
    samples = []
    for _, row in df.iterrows():
        src = str(row['transliteration'])
        conv = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"/no_think\nTranslate Akkadian to English:\n{src}"},
        ]
        if is_train:
            tgt = str(row['translation'])
            conv.append({"role": "assistant", "content": tgt})
        samples.append({"conversations": conv})
    return Dataset.from_list(samples)

train_dataset = create_conversations(all_train, is_train=True)
val_dataset = create_conversations(val_split, is_train=True)

print(f"训练样本示例:\n{train_dataset[0]}")

# ============================================================
# 4. 加载 Qwen3-8B + LoRA
# ============================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=Config.MODEL_NAME,
    load_in_4bit=Config.LOAD_IN_4BIT,
    max_lora_rank=Config.LORA_RANK,
    max_seq_length=Config.MAX_SEQ_LENGTH,
    gpu_memory_utilization=0.9,
)
tokenizer.padding_side = "left"

model = FastLanguageModel.get_peft_model(
    model,
    r=Config.LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=Config.LORA_ALPHA,
    lora_dropout=Config.LORA_DROPOUT,
    use_gradient_checkpointing="unsloth",
    random_state=Config.SEED,
)
model.config.use_cache = False

# ============================================================
# 5. 格式化
# ============================================================
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print(f"\n格式化后样本:\n{train_dataset[0]['text'][:500]}...")

# ============================================================
# 6. SFT 训练
# ============================================================
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRAD_ACCUM,
        warmup_steps=Config.WARMUP_STEPS,
        num_train_epochs=Config.EPOCHS,
        learning_rate=Config.LR,
        logging_steps=5,
        optim="adamw_torch",
        weight_decay=Config.WEIGHT_DECAY,
        label_smoothing_factor=0.1,
        lr_scheduler_type="cosine",
        seed=Config.SEED,
        report_to="none",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        save_total_limit=2,
        max_seq_length=Config.MAX_SEQ_LENGTH,
    ),
)

sample_text = train_dataset[0]['text']
print("=" * 60)
print("实际文本格式:")
print(sample_text[:800])
print("=" * 60)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("\n开始训练...")
trainer_stats = trainer.train()

# ============================================================
# 7. 保存模型
# ============================================================
model.save_pretrained(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"\n模型已保存至 {Config.OUTPUT_DIR}")
print(f"最终 train loss: {trainer_stats.training_loss:.4f}")

# ============================================================
# 8. Batch 推理评估
# ============================================================
print("\n" + "=" * 60)
print("开始验证集评估 (Batch 推理)...")
print("=" * 60)

FastLanguageModel.for_inference(model)
model.config.use_cache = True

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

hypotheses, references = [], []
total = len(val_split)
EVAL_BATCH_SIZE = 8

for i in range(0, total, EVAL_BATCH_SIZE):
    batch_df = val_split.iloc[i : i + EVAL_BATCH_SIZE]
    if i % 40 == 0:
        print(f"  进度: {i}/{total}")

    batch_messages = [[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"/no_think\nTranslate Akkadian to English:\n{row['transliteration']}"},
    ] for _, row in batch_df.iterrows()]

    batch_texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    inputs = tokenizer(
        batch_texts, return_tensors="pt", padding=True,
        truncation=True, max_length=Config.MAX_SEQ_LENGTH
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    batch_preds = tokenizer.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    for pred, (_, row) in zip(batch_preds, batch_df.iterrows()):
        pred = re.sub(r'<think>.*?</think>', '', pred, flags=re.DOTALL).strip()
        hypotheses.append(pred)
        references.append(str(row['translation']))

bleu = sacrebleu.corpus_bleu(hypotheses, [references]).score
chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2).score
geo_mean = math.sqrt(bleu * chrf) if bleu > 0 and chrf > 0 else 0.0

print(f"\n{'=' * 60}")
print(f"验证集结果 ({total} 条)")
print(f"  BLEU:    {bleu:.2f}")
print(f"  chrF++:  {chrf:.2f}")
print(f"  GeoMean: {geo_mean:.2f}")
print(f"{'=' * 60}")

for i in range(min(5, total)):
    print(f"\n  SRC: {val_split.iloc[i]['transliteration'][:100]}")
    print(f"  REF: {references[i][:100]}")
    print(f"  HYP: {hypotheses[i][:100]}")

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import torch
import re
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ============================================================
# Configuration
# ============================================================
BASE_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
ADAPTER_PATH = "/kaggle/input/notebooks/minminandy/pseudo-of-qwen3-8b-train-augment-with-extranew/qwen3-akkadian-lora"
TEST_CSV = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"
BATCH_SIZE = 4

SYSTEM_PROMPT = "You are a professional Akkadian translation expert. Translate the given Akkadian transliteration into English accurately."

# ============================================================
# Step 1: fp16 双卡加载 + LoRA
# ============================================================
print("1. Loading base model in fp16 (dual T4)...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

print(f"Base model loaded. GPU memory: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB (GPU0), {torch.cuda.memory_allocated(1)/1024**3:.1f} GB (GPU1)")

print("2. Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model ready.")

# ============================================================
# Step 2: Preprocessing
# ============================================================
def v3_official_clean_source(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\[x\]|\(break\)|\(large break\)|\(\d+ broken lines\)', '<gap>', text)
    text = re.sub(r'\b[xX]\b|\.\.\.', '<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    while '<gap> <gap>' in text:
        text = text.replace('<gap> <gap>', '<gap>')
    text = text.replace('Ḫ', 'H').replace('ḫ', 'h')
    text = text.replace('KÙ.B.', 'KÙ.BABBAR')
    sub_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    text = text.translate(sub_map)
    fractions_map = {
        '0.5': '½', '0.25': '¼', '0.3333': '⅓', '0.8333': '⅚',
        '0.625': '⅝', '0.6666': '⅔', '0.75': '¾', '0.1666': '⅙'
    }
    for dec, frac in fractions_map.items():
        text = text.replace(dec, frac)
    return text.strip()


def v3_official_clean_target(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = text.strip()
    text = text.replace('fem.', '').replace('sing.', '').replace('pl.', '')
    text = text.replace('plural', '').replace('(?)', '').replace('..', '')
    text = text.replace('?', '')
    text = re.sub(r'\bxx+\b|\bx\b', '', text)
    text = text.replace('<<', '').replace('>>', '')
    text = re.sub(r'<(?!gap>)[^>]*>', '', text)
    text = re.sub(r'\[x\]|\(break\)|\(large break\)|\(\d+ broken lines\)', '<gap>', text)
    text = re.sub(r'\b[xX]\b|\.\.\.', '<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    while '<gap> <gap>' in text:
        text = text.replace('<gap> <gap>', '<gap>')
    text = re.sub(r'\bPN\b', '<gap>', text)
    text = text.replace('-gold', ' pašallum gold')
    text = text.replace('-tax', ' šadduātum tax')
    text = text.replace('-textiles', ' kutānum textiles')
    weight_replacements = {
        '1 / 12 (shekel)': '15 grains',
        '5 / 12 shekel': '⅓ shekel 15 grains',
        '5 11 / 12 shekels': '6 shekels less 15 grains',
        '7 / 12 shekel': '½ shekel 15 grains'
    }
    for old_w, new_w in weight_replacements.items():
        text = text.replace(old_w, new_w)
    fractions_map = {
        '0.5': '½', '0.25': '¼', '0.3333': '⅓', '0.8333': '⅚',
        '0.625': '⅝', '0.6666': '⅔', '0.75': '¾', '0.1666': '⅙'
    }
    for dec, frac in fractions_map.items():
        text = text.replace(dec, frac)
    months_map = {
        'Month I': 'Month 1', 'Month II': 'Month 2', 'Month III': 'Month 3',
        'Month IV': 'Month 4', 'Month V': 'Month 5', 'Month VI': 'Month 6',
        'Month VII': 'Month 7', 'Month VIII': 'Month 8', 'Month IX': 'Month 9',
        'Month X': 'Month 10', 'Month XI': 'Month 11', 'Month XII': 'Month 12'
    }
    for rom, arab in months_map.items():
        text = text.replace(rom, arab)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return ""
    if text[0].islower():
        text = text[0].upper() + text[1:]
    if text[-1] not in ['.', '!', '?', '"', '\'', '>']:
        text += '.'
    return text


def postprocess_translation(text):
    if not isinstance(text, str) or not text.strip():
        return "broken text."
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    if not text:
        return "broken text."
    text = v3_official_clean_target(text)
    if not text:
        return "broken text."
    return text


# ============================================================
# Step 3: Prepare prompts
# ============================================================
print("--- Preparing prompts ---")
test_df = pd.read_csv(TEST_CSV)

messages_list = []
for text in test_df['transliteration']:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"/no_think\nTranslate Akkadian to English:\n{v3_official_clean_source(text)}"},
    ]
    messages_list.append(messages)

raw_prompts = [
    tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    for messages in messages_list
]

print("Prompt example:")
print(raw_prompts[0][:500])
print("=" * 60)

# ============================================================
# Step 4: Batched inference
# ============================================================
print("--- Starting inference ---")

eos_id = tokenizer.eos_token_id
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_ids = list(set(tid for tid in [eos_id, im_end_id] if tid is not None))
print(f"Stop token IDs: {stop_ids}")

# Sort by length for efficient batching
indexed_prompts = list(enumerate(raw_prompts))
indexed_prompts.sort(key=lambda x: len(x[1]))

predictions_indexed = []

for i in tqdm(range(0, len(indexed_prompts), BATCH_SIZE)):
    batch = indexed_prompts[i: i + BATCH_SIZE]
    batch_indices = [b[0] for b in batch]
    batch_prompts = [b[1] for b in batch]

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=stop_ids,
        )

    input_len = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_len:]
    decoded_batch = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

    for idx, text in zip(batch_indices, decoded_batch):
        predictions_indexed.append((idx, text))

# Restore original order
predictions_indexed.sort(key=lambda x: x[0])
predictions = [text for _, text in predictions_indexed]

# ============================================================
# Step 5: Post-processing + Submission
# ============================================================
print("--- Post-processing ---")
final_predictions = [postprocess_translation(text) for text in predictions]

for i in range(min(5, len(final_predictions))):
    print(f"\nSRC: {test_df.iloc[i]['transliteration'][:100]}")
    print(f"OUT: {final_predictions[i][:100]}")

submission_df = pd.DataFrame({
    "id": test_df["id"],
    "translation": final_predictions
})
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSubmission saved to {OUTPUT_CSV}")
print(f"Total predictions: {len(final_predictions)}")

In [ ]:
#轮询逻辑
  #step1：只用 orig + extra_clean训练基准模型
  #step2：用基准模型预测额外的古亚述语获取presudo label（利用强大的llm把控质量）
  #step3：用presudo label data + orig + extra_clean训练专家模型

#用ocr数据提取阿卡德原文做pseudo label 预测
